In [3]:
import cv2
import numpy as np
from IPython import display

display.clear_output()

In [ ]:
from ultralytics import YOLO
import cv2
import torch

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model = YOLO("yolo11n.pt")

# перший кадр тестового відео (python download_samples.py)
cap = cv2.VideoCapture("data/samples/people-walking.mp4")
ok, img = cap.read()
cap.release()
assert ok, "немає data/samples/people-walking.mp4 — запусти python download_samples.py"

result = model.predict(source=img.copy(), imgsz=320, conf=0.3, verbose=False, device=device)

for box, cnf, name in zip(result[0].boxes.xyxy, result[0].boxes.conf, result[0].boxes.cls):
    x1, y1, x2, y2 = box.cpu().numpy().astype(int)
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(img, f"Confidence: {cnf.cpu().numpy():.2f}, Class: {result[0].names[int(name.cpu().numpy())]}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)

ok = cv2.imwrite("output.jpg", img)
print(f"{device}: {len(result[0].boxes)} детекцій, output.jpg збережено: {ok}")